In [0]:
sales_orders_df = spark.read.table("ecommerce_analytics.silver.sales_orders")
order_products_df = spark.read.table("ecommerce_analytics.silver.order_products")
promotions_df = spark.read.table("ecommerce_analytics.silver.promotions")
customers_df = spark.read.table("ecommerce_analytics.silver.customers")
clicked_items_df = spark.read.table("ecommerce_analytics.silver.clicked_items")
sales_df = spark.read.table("ecommerce_analytics.silver.sales")

### WHY ONLY ONE TABLE?
 ------------------------------------------------------------
* Dimension customer stores customer master data.
* customers_df already has customer attributes.
* We do NOT need sales/orders tables because:
* Those are transaction tables.
* Dimension tables should come from master/source table.
 
### STEP 1: Start from customers_df
 =================================

In [0]:
cust = customers_df

In [0]:
from pyspark.sql.functions import *
cust = cust.withColumn("full_name", concat_ws(" ", col("first_name"), col("last_name")))


####Unix timestamp

Unix timestamp is the number of seconds elapsed since 1970-01-01 UTC. It is commonly used for efficient date/time storage and system interoperability.

In [0]:
cust = cust.withColumn(
    "created_date",
    to_date(from_unixtime(col("valid_from")))
        
)

In [0]:
cust.display()

In [0]:
cust = cust.withColumn(
    "customer_type",
    when(col("loyalty_segment") >= 3, "Premium")
    .when(col("loyalty_segment") == 2, "Regular")
    .otherwise("New")
)

In [0]:
#  want only DATE from last_update_ts
# Example: 2026-04-11

dim_customer = cust.select(
    col("customer_id"),
    col("first_name"),
    col("last_name"),
    col("full_name"),
    col("loyalty_segment"),
    col("customer_type"),
    col("created_date"),

    to_date(col("last_update_ts")).alias("last_updated")
).dropDuplicates(["customer_id"])



In [0]:
dim_customer.display()